# Stage 2 — Information Extraction

Runs IE on the **latest admission note** per patient. Prior admissions are passed as **history**.

**LLM:** `LLM_PROVIDER` in `pipeline_config.py` (`openrouter` | `ollama`)

**Input:** `cohort.pkl` (1 row/patient) | **Checkpoint:** `ie_checkpoint.json`

**Next:** `stage_03_symptom_tree.ipynb`


## Setup

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    IE_CHECKPOINT_JSON,
    LLMNotAvailableError,
    LLM_REQUEST_DELAY_SECONDS,
    check_llm,
    get_llm_config,
    information_extraction_agent,
    load_cohort,
    load_ie_checkpoint,
    print_pipeline_banner,
    save_ie_checkpoint,
    save_ie_results,
    warn_if_slow_model,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
warn_if_slow_model(model_info, LLM_CONFIG.provider)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")
cohort_df = load_cohort()
print(f"Loaded cohort: {len(cohort_df)} patients (latest note + structured data each)")


Pipeline mode : FULL (15 patients)
LLM provider  : OpenRouter (qwen/qwen-2.5-7b-instruct)
Qwen pair     : Local equivalent: ollama pull qwen2.5:7b
Admissions/patient (min): 2
Data dir      : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data
Export dir    : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
LLM ready — openrouter: qwen/qwen-2.5-7b-instruct
Timeout: 600s | note chars: 4000
Loaded cohort: 50 admissions, 15 patients


## Run Information Extraction

In [3]:
import time

checkpoint = load_ie_checkpoint()
rows = list(checkpoint.values())
done_hadm = set(checkpoint.keys())
print(f"Resuming: {len(done_hadm)} already done, {len(cohort_df) - len(done_hadm)} remaining")

for _, row in cohort_df.iterrows():
    hadm_id = int(row["hadm_id"])
    if hadm_id in done_hadm:
        continue

    history = row.get("admission_history") or []
    print(f"IE patient={row['patient_id']} hadm_id={hadm_id} ({len(history)} prior admissions)...")
    try:
        ctx = row.get("clinical_context_text") or ""
        extracted = information_extraction_agent(
            row["clinical_note"],
            config=LLM_CONFIG,
            admission_history=history,
            clinical_context_text=ctx,
        )
    except (TimeoutError, LLMNotAvailableError) as exc:
        save_ie_checkpoint(rows)
        raise type(exc)(
            f"{exc}\nCheckpoint saved ({len(rows)} patients) → {IE_CHECKPOINT_JSON}\n"
            "Re-run this cell to resume."
        ) from exc

    method = extracted.pop("_method", "unknown")
    record = {
        "patient_id": row["patient_id"],
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": hadm_id,
        "note_type": row["note_type"],
        "n_prior_admissions": len(history),
        "admission_history": history,
        "clinical_context_text": ctx,
        "structured_vitals": row.get("structured_vitals") or [],
        "structured_labs": row.get("structured_labs") or [],
        "structured_reports": row.get("structured_reports") or [],
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "ground_truth_icd10": row["ground_truth_icd10"],
        "ground_truth_dx_titles": row["ground_truth_dx_titles"],
        "n_diagnoses": int(row["n_diagnoses"]),
        "extraction_method": method,
        "symptom_count": len(extracted.get("symptoms", [])),
        "diagnosis_count": len(extracted.get("diagnoses_mentioned", [])),
        "extracted": extracted,
    }
    rows.append(record)
    done_hadm.add(hadm_id)
    save_ie_checkpoint(rows)

    if LLM_REQUEST_DELAY_SECONDS > 0:
        time.sleep(LLM_REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(rows)
print(f"\nDone — {len(results_df)} patients (latest admission each)")
results_df[["patient_id", "admission_id", "n_prior_admissions", "extraction_method", "symptom_count"]]



Resuming: 0 already done, 50 remaining
IE hadm_id=28540049 (subject_id=10361982)...
IE hadm_id=24286431 (subject_id=10361982)...
IE hadm_id=25245894 (subject_id=10426859)...


LLMNotAvailableError: API error 429: {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"qwen/qwen-2.5-7b-instruct is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations","provider_name":"Phala","is_byok":false}},"user_id":"user_3AXfnPm8nGPO0Th6JNIWQopCqjM"}

## Save Results

In [ ]:
out = save_ie_results(results_df)
print(f"Saved → {out}")